In [ ]:
# Step 1: Install gdown
# Run this command in your terminal
%pip install gdown

# Step 2: Import the gdown module
import gdown

# Step 3: Define the file ID and the destination path
file_id = '16TNcpCXUGwBwtm4q-HMdxyCbUfEv1yw-'
destination = 'downloaded_file'

# Step 4: Construct the download URL
url = f'https://drive.google.com/uc?id={file_id}'

# Step 5: Download the file
gdown.download(url, destination, quiet=False)

In [ ]:
import torch
from weakly_supervised_mVLP.albef.modeling.model_pretrain import ALBEF_Stage1


model = ALBEF_Stage1()
en_text = "The red cat is playing with the boy in the park."
vi_text = "Chú mèo màu đỏ đang chơi với cậu bé ở công viên."
tokenizer = model.tokenizer
def encode_text(text, device='cuda'):
    model.to(device)
    with torch.no_grad():
        hidden_state = model.text_encoder.bert(**tokenizer(text, return_tensors='pt', padding='max_length', truncation=True, max_length=40).to(device), mode='text').last_hidden_state
        cls_token = hidden_state[:, 0, :]
        text_feat = model.text_proj(cls_token)
        norm_text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)
    return norm_text_feat

eng_emb = encode_text(en_text)
vie_emb = encode_text(vi_text)
eng_emb @ vie_emb.T